In [1]:
import pandas as pd


In [2]:
test = pd.read_csv("../../../sample_submission.csv")
test.drop(columns=['Revenue', 'COGS'], inplace=True)
test


,Date
0,2023-01-01
1,2023-01-02
2,2023-01-03
3,2023-01-04
4,2023-01-05
...,...
543,2024-06-27
544,2024-06-28
545,2024-06-29
546,2024-06-30


In [3]:
def mmdd_to_doy(mmdd):
    return pd.to_datetime(f"2021-{mmdd}").dayofyear  # năm bất kỳ không nhuận


In [4]:
# dữ liệu sales
test['Date'] = pd.to_datetime(test['Date'])

# chuyển sang day of year
test['doy'] = test['Date'].dt.dayofyear

# define promo ranges (MM-DD)
promo_ranges = [
    ("03-18", "04-17"),
    ("06-23", "07-22"),
    ("11-18", "01-02"),  # qua năm
    ("08-30", "10-02")
]


In [5]:
# convert range
promo_doy_ranges = [(mmdd_to_doy(s), mmdd_to_doy(e)) for s, e in promo_ranges]


In [6]:
def is_promo(doy):
    for start, end in promo_doy_ranges:
        if start <= end:
            if start <= doy <= end:
                return 1
        else:
            # case qua năm (wrap)
            if doy >= start or doy <= end:
                return 1
    return 0


In [7]:
test['is_promo'] = test['doy'].apply(is_promo)
test


,Date,doy,is_promo
0,2023-01-01,1,1
1,2023-01-02,2,1
2,2023-01-03,3,0
3,2023-01-04,4,0
4,2023-01-05,5,0
...,...,...,...
543,2024-06-27,179,1
544,2024-06-28,180,1
545,2024-06-29,181,1
546,2024-06-30,182,1


In [8]:
test['is_promo'].value_counts()


is_promo
0    364
1    184
Name: count, dtype: int64

In [9]:
test['year'] = test['Date'].dt.year
test['month'] = test['Date'].dt.month


In [10]:
test['quarter'] = test['Date'].dt.quarter
test


,Date,doy,is_promo,year,month,quarter
0,2023-01-01,1,1,2023,1,1
1,2023-01-02,2,1,2023,1,1
2,2023-01-03,3,0,2023,1,1
3,2023-01-04,4,0,2023,1,1
4,2023-01-05,5,0,2023,1,1
...,...,...,...,...,...,...
543,2024-06-27,179,1,2024,6,2
544,2024-06-28,180,1,2024,6,2
545,2024-06-29,181,1,2024,6,2
546,2024-06-30,182,1,2024,6,2


In [11]:
def get_season(month):
    if month in [2, 3, 4]:
        return 'Spring'
    elif month in [5, 6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Autumn'
    else:
        return 'Winter'  # 12, 1

test['season'] = test['month'].apply(get_season)


In [12]:
labels = {'Autumn': 0, 'Spring': 1, 'Summer': 2, 'Winter': 3}
test['season'] = test['season'].map(labels)
test


,Date,doy,is_promo,year,month,quarter,season
0,2023-01-01,1,1,2023,1,1,3
1,2023-01-02,2,1,2023,1,1,3
2,2023-01-03,3,0,2023,1,1,3
3,2023-01-04,4,0,2023,1,1,3
4,2023-01-05,5,0,2023,1,1,3
...,...,...,...,...,...,...,...
543,2024-06-27,179,1,2024,6,2,2
544,2024-06-28,180,1,2024,6,2,2
545,2024-06-29,181,1,2024,6,2,2
546,2024-06-30,182,1,2024,6,2,2


In [13]:
test.to_csv("test_prepared_1.csv", index=False)


In [14]:
test.drop(columns=['Date'], inplace=True)


In [15]:
import joblib
rev_model = joblib.load("../rev_model.pkl")
cogs_model = joblib.load("../cogs_model.pkl")


In [16]:
submit = pd.read_csv("../../../sample_submission.csv")
submit['Revenue'] = rev_model.predict(test)
submit['COGS'] = cogs_model.predict(test)
submit.to_csv("submission.csv", index=False)
